# 05. Model Optimization & Dynamic Quantization

- **Role/Owner:** Member 2 (ML & Data Engineering Lead)
- **Core Responsibility:** Apply PyTorch INT8 dynamic quantization to fine-tuned DistilBERT to reduce model memory footprint to <200MB with minimal accuracy degradation.
- **Key Interface/Contract:** Consumes `distilbert_finetuned`; outputs `backend/models/distilbert_quantized.pt` and size/latency benchmark logs.

**Size target, corrected:** the original header aimed for about 65 MB, which assumed every weight becomes INT8. Dynamic quantization converts only the `Linear` layers; the token and position embeddings (about 23.8M parameters, roughly 95 MB) stay FP32. The measured result is therefore about 139 MB, which still meets the < 200 MB requirement.

**Why dynamic quantization:**
- It needs no calibration data: weights are converted to INT8 ahead of time, and activations are quantized on the fly per batch.
- In DistilBERT the `Linear` layers (attention projections and feed-forward blocks) do most of the compute, so converting them gives most of the speed-up.
- INT8 kernels run on CPU only. Both models are benchmarked on CPU with the `qnnpack` engine, the only quantized engine on Apple Silicon and the one the API server uses.

**Inputs:** `backend/models/distilbert_finetuned/` from notebook 04, `backend/data/processed/test.csv`.

**Outputs:**
- `backend/models/distilbert_quantized.pt`: INT8 `state_dict` (gitignored; shared as GitHub Release `model-v2`).
- `backend/models/distilbert_tokenizer/`: tokenizer files and `config.json`, no weights (committed).
- `backend/reports/quantization_benchmark.md`: size, latency and accuracy, FP32 vs INT8.

In [1]:
# Role/Owner: Member 2 (ML & Data Engineering Lead)
# Core Responsibility: PyTorch INT8 dynamic quantization for memory optimization
# Key Interface/Contract: Input: fine-tuned model -> Output: backend/models/distilbert_quantized.pt

# torch is imported before sklearn on purpose: in ml-env both ship their own libomp, and loading sklearn's first
# makes torch's CPU kernels segfault.
import torch

import hashlib
import json
import platform
import statistics
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import transformers
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

sys.path.insert(0, '..')  # make backend/ importable so the service module is the single source of truth
from app.services.inference import TOKENIZER_DIR_NAME, _load_distilbert, quantize_int8

SEED = 42
MAX_LENGTH = 256
BATCH_SIZE = 16
LATENCY_WARMUP, LATENCY_SINGLE_RUNS, LATENCY_BATCH_RUNS, LATENCY_BATCH_SIZE = 5, 50, 10, 32
MAX_SIZE_MB, MAX_F1_DROP = 200, 0.02

MODELS_DIR = Path('../models')
FINETUNED_DIR = MODELS_DIR / 'distilbert_finetuned'
QUANTIZED_PATH = MODELS_DIR / 'distilbert_quantized.pt'
TOKENIZER_DIR = MODELS_DIR / TOKENIZER_DIR_NAME
REPORT_PATH = Path('../reports/quantization_benchmark.md')

torch.manual_seed(SEED)
DEVICE = torch.device('cpu')  # INT8 kernels are CPU-only; FP32 runs on CPU too for a fair comparison
print(f'torch {torch.__version__} | transformers {transformers.__version__} | '
      f'engines {torch.backends.quantized.supported_engines} | threads {torch.get_num_threads()}')

/opt/anaconda3/envs/ml-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.14.1 | transformers 5.18.0 | engines ['qnnpack'] | threads 4


`quantize_int8` and `_load_distilbert` are imported from `app/services/inference.py`, so this notebook measures exactly what the API loads. `quantize_int8` sets the `qnnpack` engine and silences only the known `torch.ao.quantization` deprecation notice: torch 2.14.1 still ships the eager-mode API but points to `torchao` (`quantize_` with an INT8 dynamic config) as the migration path. Do not upgrade torch in `ml-env` without re-running this notebook.

## Part 1: Quantize

In [2]:
fp32_model = AutoModelForSequenceClassification.from_pretrained(FINETUNED_DIR).to(DEVICE).eval()
tokenizer = AutoTokenizer.from_pretrained(FINETUNED_DIR)
int8_model = quantize_int8(AutoModelForSequenceClassification.from_pretrained(FINETUNED_DIR).eval())

quantized_linear = torch.ao.nn.quantized.dynamic.Linear
n_converted = sum(isinstance(m, quantized_linear) for m in int8_model.modules())
n_fp32_linear = sum(isinstance(m, torch.nn.Linear) for m in fp32_model.modules())
print(f'Engine: {torch.backends.quantized.engine}')
print(f'Linear layers converted to INT8: {n_converted} of {n_fp32_linear}')
print('Still FP32:', [name for name, m in int8_model.named_modules() if isinstance(m, torch.nn.Embedding)])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 12066.27it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 25299.13it/s]

/opt/anaconda3/envs/ml-env/lib/python3.11/site-packages/torch/ao/nn/quantized/modules/utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:116.)
  qweight = torch.quantize_per_tensor(


Engine: qnnpack
Linear layers converted to INT8: 38 of 38
Still FP32: ['distilbert.embeddings.word_embeddings', 'distilbert.embeddings.position_embeddings']


## Part 2: Size

Each `state_dict` is serialised with `torch.save` and measured on disk, which is what the API loads.

In [3]:
def saved_size_mb(model: torch.nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix='.pt') as f:
        torch.save(model.state_dict(), f.name)
        return Path(f.name).stat().st_size / 1e6


fp32_mb, int8_mb = saved_size_mb(fp32_model), saved_size_mb(int8_model)
embedding_params = sum(m.weight.numel() for m in int8_model.modules() if isinstance(m, torch.nn.Embedding))
print(f'FP32 {fp32_mb:.1f} MB | INT8 {int8_mb:.1f} MB | {fp32_mb / int8_mb:.2f}x smaller')
print(f'FP32 embeddings kept in the INT8 model: {embedding_params / 1e6:.1f}M parameters, about {embedding_params * 4 / 1e6:.0f} MB')

FP32 267.9 MB | INT8 138.7 MB | 1.93x smaller
FP32 embeddings kept in the INT8 model: 23.8M parameters, about 95 MB


## Part 3: Accuracy on the test split

Same tokenization as notebook 04 (truncation at 256 tokens, dynamic padding, batches of 16). The FP32 model is re-scored here on CPU so both numbers come from the same device.

In [4]:
test_df = pd.read_csv('../data/processed/test.csv')
test_df['text_clean'] = test_df['text_clean'].fillna('')
assert len(test_df) == 953
encodings = tokenizer(test_df['text_clean'].tolist(), truncation=True, max_length=MAX_LENGTH)
test_items = [{key: encodings[key][i] for key in encodings} for i in range(len(test_df))]
test_loader = DataLoader(test_items, batch_size=BATCH_SIZE, collate_fn=DataCollatorWithPadding(tokenizer))


@torch.inference_mode()
def predict(model: torch.nn.Module) -> np.ndarray:
    return np.concatenate([model(**batch).logits.argmax(dim=-1).numpy() for batch in test_loader])


y_test = test_df['label'].to_numpy()
predictions = {'FP32': predict(fp32_model), 'INT8': predict(int8_model)}
accuracy_df = pd.DataFrame([
    {'Model': name, 'Accuracy': accuracy_score(y_test, pred),
     'Precision (macro)': precision_score(y_test, pred, average='macro'),
     'Recall (macro)': recall_score(y_test, pred, average='macro'),
     'Macro F1': f1_score(y_test, pred, average='macro')}
    for name, pred in predictions.items()])
fp32_f1, int8_f1 = accuracy_df['Macro F1']
agreement = (predictions['FP32'] == predictions['INT8']).mean()

reference_f1 = json.loads(Path('../reports/distilbert_metrics.json').read_text())['macro_f1']
display(accuracy_df.round(4))
print(f'Macro F1 change: {(int8_f1 - fp32_f1) * 100:+.2f} points | prediction agreement {agreement:.2%}')
print(f'Notebook 04 test Macro F1 ({reference_f1:.4f}) vs FP32 on CPU here ({fp32_f1:.4f})')

[W1023 09:24:22.383778000 qlinear_dynamic.cpp:251] Warning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (function operator())


,Model,Accuracy,Precision (macro),Recall (macro),Macro F1
0,FP32,0.9444,0.8971,0.8434,0.8675
1,INT8,0.9433,0.9146,0.8186,0.8579


Macro F1 change: -0.96 points | prediction agreement 98.43%
Notebook 04 test Macro F1 (0.8675) vs FP32 on CPU here (0.8675)


## Part 4: Latency

`torch.inference_mode()` at torch's default CPU thread count (recorded in the report). After 5 warm-up calls, the median of 50 single-claim calls (the API's case) and of 10 calls on a batch of 32 test texts. Both models get identical inputs.

In [5]:
single_input = tokenizer(test_df['text_clean'].iloc[0], truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
batch_input = tokenizer(test_df['text_clean'].iloc[:LATENCY_BATCH_SIZE].tolist(), truncation=True,
                        max_length=MAX_LENGTH, padding=True, return_tensors='pt')


@torch.inference_mode()
def median_ms(model: torch.nn.Module, inputs, runs: int) -> float:
    for _ in range(LATENCY_WARMUP):
        model(**inputs)
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        model(**inputs)
        times.append((time.perf_counter() - start) * 1000)
    return statistics.median(times)


latency_df = pd.DataFrame([
    {'Model': name,
     'Single claim (ms)': median_ms(model, single_input, LATENCY_SINGLE_RUNS),
     f'Batch of {LATENCY_BATCH_SIZE} (ms)': median_ms(model, batch_input, LATENCY_BATCH_RUNS)}
    for name, model in (('FP32', fp32_model), ('INT8', int8_model))])
speedup = latency_df.iloc[0, 1:] / latency_df.iloc[1, 1:]
display(latency_df.round(1))
print('INT8 speed-up:', ', '.join(f'{col} {value:.2f}x' for col, value in speedup.items()))

,Model,Single claim (ms),Batch of 32 (ms)
0,FP32,15.3,125.3
1,INT8,5.6,338.5


INT8 speed-up: Single claim (ms) 2.73x, Batch of 32 (ms) 0.37x


## Part 5: Gate

The artifact must be under 200 MB and lose at most 2 Macro F1 points. If the F1 gate fails, stop and record the numbers rather than loosening it; FP16 is the fallback named in the project plan.

In [6]:
gates = {f'INT8 size < {MAX_SIZE_MB} MB': int8_mb < MAX_SIZE_MB,
         f'Macro F1 drop <= {MAX_F1_DROP * 100:.0f} points': fp32_f1 - int8_f1 <= MAX_F1_DROP}
for name, passed in gates.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert int8_mb < MAX_SIZE_MB, f'INT8 artifact is {int8_mb:.1f} MB'
assert fp32_f1 - int8_f1 <= MAX_F1_DROP, f'Macro F1 dropped {(fp32_f1 - int8_f1) * 100:.2f} points'

PASS: INT8 size < 200 MB
PASS: Macro F1 drop <= 2 points


## Part 6: Export and round-trip check

The quantized model is saved as a `state_dict`, not a pickled module, so it loads with `torch.load(..., weights_only=True)` (no code is unpickled) and survives `transformers` upgrades. At runtime the API needs only this file plus the small `distilbert_tokenizer/` folder (tokenizer and `config.json`). The round trip below loads it through the API's own `_load_distilbert` and checks the logits match.

In [7]:
torch.save(int8_model.state_dict(), QUANTIZED_PATH)
tokenizer.save_pretrained(TOKENIZER_DIR)
fp32_model.config.save_pretrained(TOKENIZER_DIR)
assert not list(TOKENIZER_DIR.glob('*.safetensors')) and not list(TOKENIZER_DIR.glob('*.bin'))

reloaded_model, reloaded_tokenizer = _load_distilbert(QUANTIZED_PATH, TOKENIZER_DIR)
with torch.inference_mode():
    assert torch.allclose(int8_model(**batch_input).logits, reloaded_model(**batch_input).logits)
assert reloaded_tokenizer(test_df['text_clean'].iloc[0])['input_ids'] == single_input['input_ids'][0].tolist()

sha256 = hashlib.sha256(QUANTIZED_PATH.read_bytes()).hexdigest()
artifact_mb = QUANTIZED_PATH.stat().st_size / 1e6
print(f'Round trip OK. {QUANTIZED_PATH} ({artifact_mb:.1f} MB), SHA-256 {sha256}')
print(f'{TOKENIZER_DIR}: {sorted(p.name for p in TOKENIZER_DIR.iterdir())}')

/opt/anaconda3/envs/ml-env/lib/python3.11/site-packages/torch/_utils.py:466: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,


Round trip OK. ../models/distilbert_quantized.pt (138.7 MB), SHA-256 47c2fbf256d2f760156bf3a00cd9f1b71b110162aa9d189647084abc640c943e
../models/distilbert_tokenizer: ['config.json', 'tokenizer.json', 'tokenizer_config.json']


In [8]:
def md_table(df: pd.DataFrame, floatfmt: str = '.3f') -> str:
    header = '| ' + ' | '.join(map(str, df.columns)) + ' |'
    sep = '|' + '|'.join('---' for _ in df.columns) + '|'
    fmt = lambda v: format(v, floatfmt) if isinstance(v, float) else (f'{v:,}' if isinstance(v, (int, np.integer)) else str(v))
    return '\n'.join([header, sep] + ['| ' + ' | '.join(fmt(v) for v in row) + ' |' for row in df.itertuples(index=False)])


try:
    chip = subprocess.run(['sysctl', '-n', 'machdep.cpu.brand_string'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    chip = ''
single_speedup, batch_speedup = speedup.iloc[0], speedup.iloc[1]
latency_note = ('' if min(single_speedup, batch_speedup) >= 1.1 else
                f"On this machine INT8 saves memory rather than time: single-claim latency changes by {single_speedup:.2f}x and "
                f"batched latency by {batch_speedup:.2f}x. FP32 matrix multiplies run on the platform's optimised BLAS, while "
                f"`{torch.backends.quantized.engine}` dynamic INT8 kernels also quantize activations on every call; x86 servers "
                f"using `fbgemm` may behave differently (not measured here). The API scores one claim per request, where the "
                f"two models are close.\n")
size_df = pd.DataFrame([{'Model': 'FP32', 'Size (MB)': fp32_mb}, {'Model': 'INT8', 'Size (MB)': int8_mb}])
gate_df = pd.DataFrame([{'Gate': name, 'Result': 'pass' if passed else 'fail'} for name, passed in gates.items()])
env_df = pd.DataFrame([
    ('torch', torch.__version__), ('transformers', transformers.__version__),
    ('Quantized engine', torch.backends.quantized.engine), ('Threads', str(torch.get_num_threads())),
    ('Machine', chip or platform.processor() or platform.machine()), ('Python', platform.python_version()),
], columns=['Item', 'Value'])

report = f"""# Quantization Benchmark: FP32 vs INT8 DistilBERT

Generated by `backend/notebooks/05_quantization.ipynb`. Re-run the notebook to refresh it.

PyTorch dynamic quantization (`quantize_dynamic`, `{{nn.Linear}}` to `qint8`) of the fine-tuned model from notebook 04.
{n_converted} of {n_fp32_linear} `Linear` layers were converted; the embeddings ({embedding_params / 1e6:.1f}M parameters) stay FP32,
which is why the artifact is about {int8_mb:.0f} MB rather than the 65 MB once estimated. Everything below runs on CPU.

## Size

{md_table(size_df, '.1f')}

INT8 is {fp32_mb / int8_mb:.2f}x smaller.

## Accuracy (test split, {len(test_df):,} rows)

{md_table(accuracy_df, '.4f')}

Macro F1 change: {(int8_f1 - fp32_f1) * 100:+.2f} points. The two models agree on {agreement:.2%} of test predictions.

## Latency (median, CPU)

{md_table(latency_df, '.1f')}

INT8 speed-up: {', '.join(f'{col.replace(" (ms)", "").lower()} {value:.2f}x' for col, value in speedup.items())}.
{LATENCY_WARMUP} warm-up calls, then {LATENCY_SINGLE_RUNS} single-claim calls and {LATENCY_BATCH_RUNS} batch calls per model.
{latency_note}

## Gates

{md_table(gate_df)}

## Artifact

- `backend/models/distilbert_quantized.pt`: {artifact_mb:.1f} MB, SHA-256 `{sha256}`
- `backend/models/distilbert_tokenizer/`: tokenizer and `config.json`, no weights

The `.pt` file is a `state_dict`; load it with `app.services.inference._load_distilbert`, which rebuilds the
quantized skeleton from `config.json` and calls `torch.load(..., weights_only=True)`. See `docs/model-artifacts.md`.

## Environment

{md_table(env_df)}

`torch.ao.quantization` prints a deprecation notice on torch 2.14.1 pointing to `torchao`; it still works, and the
API depends on it, so torch should not be upgraded without re-running this notebook.
"""
REPORT_PATH.write_text(report)
print(f'Wrote {REPORT_PATH} ({len(report.splitlines())} lines)')

Wrote ../reports/quantization_benchmark.md (66 lines)
